In [1]:
import json, os, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from astropy.io import fits
from IPython.display import display, Image

# Data

## Animation

## Light Curves

In [5]:
display(pd.DataFrame([
    ("time",    "days"),
    ("signal",  "magnitude"),
    ("dsignal", "error (placeholder: constant ~22.2)"),
], columns=["column", "meaning"]).set_index("column"))

files = ["mock_data/testCAM-i_LC_sampled 98.json", "mock_data/testCAM-i_LC_sampled 99.json"]
lcs = {f: [{k: np.asarray(v, float) for k, v in e.items()} for e in json.load(open(f))]
       for f in files}

display(pd.DataFrame([{"file": f, "image": i, "N": len(c["time"]),
                       "time": f'{c["time"].min():.0f} - {c["time"].max():.0f}',
                       "signal": f'{c["signal"].min():.2f} - {c["signal"].max():.2f}',
                       "dsignal": round(c["dsignal"].mean(), 3)}
                      for f, cs in lcs.items() for i, c in enumerate(cs)]
                     ).set_index(["file", "image"]))

print(f"{files[0]}, image 0, first rows:")
display(pd.DataFrame({k: v[:5] for k, v in lcs[files[0]][0].items()}))

,meaning
column,
time,days
signal,magnitude
dsignal,error (placeholder: constant ~22.2)


N         time         signal  \
file                                   image                                    
mock_data/testCAM-i_LC_sampled 98.json 0      707  1200 - 4287  22.73 - 23.23   
                                       1      707  1200 - 4287  25.18 - 25.57   
                                       2      707  1200 - 4287  34.81 - 35.21   
                                       3      707  1200 - 4287  23.18 - 23.58   
                                       4      707  1200 - 4287  23.79 - 24.22   
mock_data/testCAM-i_LC_sampled 99.json 0      707  1200 - 4287  23.62 - 24.05   
                                       1      707  1200 - 4287  24.05 - 25.00   
                                       2      707  1200 - 4287  34.81 - 35.21   
                                       3      707  1200 - 4287  22.60 - 23.21   
                                       4      707  1200 - 4287  22.19 - 23.14   

                                              dsignal  
file                                   image           
mock_data/testCAM-i_LC_sampled 98.json 0       22.194  
                                       1       22.202  
                                       2       22.190  
                                       3       22.198  
                                       4       22.203  
mock_data/testCAM-i_LC_sampled 99.json 0       22.200  
                                       1       22.196  
                                       2       22.190  
                                       3       22.195  
                                       4       22.194

mock_data/testCAM-i_LC_sampled 98.json, image 0, first rows:


,dsignal,signal,time
0,22.193939,22.830781,1200.00
1,22.193938,22.827793,1206.99
2,22.193938,22.859924,1214.01
3,22.193938,22.870736,1225.01
4,22.193938,22.861174,1232.98


In [ ]:
COL = {0: "C0", 1: "C1", 2: "0.6", 3: "C2", 4: "C3"}
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for ax, f in zip(axes, files):
    for i in [0, 1, 3, 4]:                    # image 2 is at ~35 mag, off scale
        ax.plot(lcs[f][i]["time"], lcs[f][i]["signal"], ".-", ms=3, lw=.6,
                color=COL[i], label=f"image {i}")
    ax.invert_yaxis(); ax.grid(alpha=.25)
    ax.set(title=f, ylabel="Mag"); ax.legend(ncol=4, fontsize=8)
axes[1].set_xlabel("t [days]")
plt.tight_layout(); plt.show()

# Methods

# Metric

In [33]:
# You do not own this
df_truth = pd.DataFrame(columns=["ID", "dt_01_days", "dt_02_days", "dt_03_days"])

df_truth.loc[len(df_truth)] = ["AAA", 0.11, "NA", "NA"]  # double system
df_truth.loc[len(df_truth)] = ["BBB", 0.13, 0.19, 0.31]  # quad system
df_truth.loc[len(df_truth)] = ["CCC", 0.13, 0.19, 0.31]  # malfomed line (ID does not exists), will be skipped by the scorer

display(df_truth)


,ID,dt_01_days,dt_02_days,dt_03_days
0,AAA,0.11,NA,NA
1,BBB,0.13,0.19,0.31
2,CCC,0.13,0.19,0.31


In [67]:
# You submit this
df_results = pd.DataFrame(columns=["ID", "dt_01_days", "dt_02_days", "dt_03_days"])

df_results.loc[len(df_results)] = ["AAA", 0.10, "NA", "NA"]   # double system
df_results.loc[len(df_results)] = ["BBB", 0.11, 0.22, 0.33]  # You do not own this# quad system

display(df_results)


,ID,dt_01_days,dt_02_days,dt_03_days
0,AAA,0.10,NaN,NaN
1,BBB,0.11,0.22,0.33


In [68]:
def score(df_results, df_truth):
    
    df_comparison = df_truth.merge(
        df_results, on="ID", suffixes=("_truth", "_result")
    )
    
    for col in ["dt_01_days", "dt_02_days", "dt_03_days"]:
        truth  = pd.to_numeric(df_comparison[f"{col}_truth"],  errors="coerce")
        result = pd.to_numeric(df_comparison[f"{col}_result"], errors="coerce")
        df_comparison[f"{col}_error"] = result - truth
    
    display(df_comparison.head())
    
    # Average across all error values, ignoring NaN
    err_mu = df_comparison.filter(like="_error").stack().mean()
    
    return err_mu

print("You score is: %.2g" % score(df_results, df_truth))

,ID,dt_01_days_truth,dt_02_days_truth,dt_03_days_truth,dt_01_days_result,dt_02_days_result,dt_03_days_result,dt_01_days_error,dt_02_days_error,dt_03_days_error
0,AAA,0.11,NA,NA,0.10,NaN,NaN,-0.01,NaN,NaN
1,BBB,0.13,0.19,0.31,0.11,0.22,0.33,-0.02,0.03,0.02


You score is: 0.005
